# Phase 3 & 4: Modellierung und Evaluierung (Cost-Sensitive Learning)

Dieses Notebook dient als Vorlage und Ausführungsumgebung für die vergleichende Modellierung auf dem PackWISE (K3I-Cycling) Datensatz. Gemäß unserer Strategie vergleichen wir:
1. **Ultralytics YOLOv8 / YOLO11** (SOTA Echtzeit-Performance)
2. **RT-DETR (PekingU/rtdetr_r50vd)** (State-of-the-Art Real-Time Transformer)
3. **facebook/detr-resnet-50** (Klassische Transformer-Baseline)

Zusätzlich entwickeln wir im letzten Abschnitt die **Business Cost-Matrix**.

In [2]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ultralytics import YOLO, RTDETR

print(f"PyTorch Version: {torch.__version__}")
print(f"MPS (Apple Silicon Support) available: {torch.backends.mps.is_available()}")

PyTorch Version: 2.11.0
MPS (Apple Silicon Support) available: True


## 1. Ultralytics YOLOv8 & YOLO11

YOLO ist enorm ressourceneffizient auf Bildern von Förderbändern. Hier trainieren und evaluieren wir ein YOLO-Modell auf unseren generierten `yolo.yaml` Daten.

In [3]:
# Pfad zur zuvor generierten yolo.yaml
data_path = "/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/yolo.yaml"

# 1. YOLOv8 Laden (Alternative: 'yolo11n.pt' für das neueste SOTA release)
model_yolo = YOLO('yolov8n.pt') 

# 2. Training starten 
model_yolo.train(data=data_path, epochs=8, imgsz=640, device="mps", workers=0, amp=False, batch=4)

# 3. Evaluierung / Validation
metrics_yolo = model_yolo.val()
print(f"YOLOv8 mAP@50: {metrics_yolo.box.map50}")

Ultralytics 8.4.41 🚀 Python-3.11.7 torch-2.11.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/yolo.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-8, nbs=64, nms=False, opset=None, optimize=False,

KeyboardInterrupt: 

## 2. RT-DETR (State-of-the-Art Transformer)

Während klassische DETRs eher langsam waren, bringt RT-DETR (`PekingU/rtdetr_r50vd`) die transformer-basierte Präzision auf Echtzeit-Niveau. Glücklicherweise unterstützt Ultralytics RT-DETR nativ! Wir können es genau wie YOLO verwenden.

In [ ]:
# RT-DETR Basismodell laden (ResNet50 Backbone - ähnlich zu rtdetr_r50vd)
model_rtdetr = RTDETR('rtdetr-l.pt')

# 1. Training (Dies wird mehr VRAM als YOLOv8 benötigen)
model_rtdetr.train(data=data_path, epochs=4, imgsz=640, device="mps", workers=0, amp=False, batch=2)

# 2. Evaluierung
metrics_rtdetr = model_rtdetr.val()
print(f"RT-DETR mAP@50: {metrics_rtdetr.box.map50}")

Ultralytics 8.4.41 🚀 Python-3.11.7 torch-2.11.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/yolo.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=Fal

RuntimeError: MPS backend out of memory (MPS allocated: 8.86 GiB, other allocations: 212.97 MiB, max allowed: 9.07 GiB). Tried to allocate 9.38 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

### Huggingface-Alternative für facebook/detr-resnet-50 & PekingU/rtdetr_r50vd
Wenn man komplett aus dem Huggingface-Ökosystem arbeiten möchte, bedient man sich der `transformers` library. *Dies erfordert aber einen eigenen Training-Loop oder die Dataset-Anpassung via HF Datasets!*

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

# Beispiel für das Laden des puren HF-Modells:
# processor = AutoImageProcessor.from_pretrained("PekingU/rtdetr_r50vd")
# hf_model_rtdetr = AutoModelForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd")

# processor_detr = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
# hf_model_detr = AutoModelForObjectDetection.from_pretrained("facebook/detr-resnet-50")

# In der Praxis (gerade in echten Business-Projekten) 
# ist der Ultralytics-Weg von Oben für SOTA-Detektoren deutlich fehlerunanfälliger!

## 3. Die Business-Evaluierung: Kostenbasierte Gewichtung (Cost-Matrix)

Standard-Metriken (mAP) behandeln alle Verwechslungen gleich. Wir greifen nun die echten Ergebnisse unseres Trainings oben (`metrics_yolo.confusion_matrix.matrix`) ab und leiten diese in eine Kosten-Funktion.

In [ ]:
def create_dummy_cost_matrix(classes):
    cost_df = pd.DataFrame(index=classes, columns=classes).fillna(0.0)
    cost_df[:] = 1.0 # 1 Einheit Störkosten für jegliche Falschklassifikation
    np.fill_diagonal(cost_df.values, 0.0)
    
    if "plastic-bottle" in classes and "paper-other" in classes:
        cost_df.loc["plastic-bottle", "paper-other"] = 25.0  # Plastik rutscht in Papiersammlung = zerstörter Papierwert
    
    if "aluminium-coated-foil" in classes and "undefined" in classes:
        cost_df.loc["aluminium-coated-foil", "undefined"] = 5.0 # Verlust in den Restmüll
        
    if "plastic-foil/bag" in classes and "paper-bag" in classes:
        cost_df.loc["plastic-foil/bag", "paper-bag"] = 10.0

    return cost_df

def evaluate_business_impact(confusion_matrix_df, cost_matrix_df):
    financial_impact_matrix = confusion_matrix_df * cost_matrix_df
    total_cost = financial_impact_matrix.sum().sum()
    return total_cost, financial_impact_matrix

# ==========================================
# EVALUATION DER ECHTEN MODELLE
# ==========================================
# Die 23 echten K3I-Klassen laut der yolo.yaml Liste
real_classes = [
    "undefined", "plastic-foil/bag", "aluminium-coated-foil", "paper-bag", 
    "beverage-carton", "plastic-other", "plastic-lid", "plastic-cup/pot", 
    "blister", "metal-can", "metal-foil", "net", "metal-lid", "tissue", 
    "wood", "paper-other", "foamed-plastic", "textile", "plastic-bottle", 
    "ribbon/tape", "metal-other", "tube", "paper-cup/pot"
]

cost_matrix = create_dummy_cost_matrix(real_classes)

try:
    # 1. YOLOv8 Evaluation
    cm_yolo_np = metrics_yolo.confusion_matrix.matrix
    # Ultralytics fügt background (hintergrund) als letzte Dimension an, also trimmen wir auf unsere 23 Klassen:
    cm_yolo_np = cm_yolo_np[:23, :23]
    cm_yolo_df = pd.DataFrame(cm_yolo_np, index=real_classes, columns=real_classes)
    yolo_loss, _ = evaluate_business_impact(cm_yolo_df, cost_matrix)
    print(f"============================")
    print(f"YOLOv8 Business Penalty: {yolo_loss} Einheiten")
    
    # 2. RT-DETR Evaluation
    cm_rtdetr_np = metrics_rtdetr.confusion_matrix.matrix
    cm_rtdetr_np = cm_rtdetr_np[:23, :23]
    cm_rtdetr_df = pd.DataFrame(cm_rtdetr_np, index=real_classes, columns=real_classes)
    rtdetr_loss, _ = evaluate_business_impact(cm_rtdetr_df, cost_matrix)
    print(f"RT-DETR Business Penalty: {rtdetr_loss} Einheiten")
    print(f"============================")
    print("\nGEWINNER IST DAS MODELL MIT DEM NIEDRIGEREN PENALTY-WERT!")
    
except NameError:
    print("INFO: Du musst erst oben die Modelle trainieren, bevor hier eine Auswertung berechnet werden kann!")